# K-SPADA Data Preparation

[Project overview](../README.md) · [01 Data preparation](./01_data_preparation.ipynb) · [02 DLPFC](./02_DLPFC.ipynb) · [03 Drosophila embryos](./03_Drosophila%20embryos.ipynb) · [04 Embryo](./04_Embryo.ipynb) · [05 Liver cancer](./05_liver_cancer.ipynb)

This tutorial prepares training, validation, and query ligand-receptor-informed similarity graph files from a generic spatial transcriptomics AnnData object.

## Workflow
1. Define paths and sample splits.
2. Validate the required AnnData fields.
3. Separate reference and query samples.
4. Build path features and ligand-receptor-informed similarity graphs.

Run this notebook from the `tutorials/` directory. Paths are relative to this directory or to the repository root (`..`). This example uses samples 4–15 for training, sample 16 for validation, and samples 17–21 for queries. The source dataset is stored at `../../../../TOSICA-main/data/liver_cancer/all_hvg_3000.h5ad`; adapt `DATA_DIR` to your own data location.


In [1]:
from pathlib import Path
import anndata as ad
import KSPADA

# PROJECT_ROOT = Path("..").resolve()
# DATA_DIR = PROJECT_ROOT / "experiment_data"
# OUTPUT_DIR = PROJECT_ROOT / "outputs" / "data_preparation"
# SOURCE_H5AD = DATA_DIR / "source_data.h5ad"

# TRAIN_SAMPLES = ["train_sample_1", "train_sample_2"]
# VALIDATION_SAMPLES = ["validation_sample"]
# QUERY_SAMPLES = ["query_sample"]
COMMUNICATION_GRAPH_NEIGHBORS = 2
FORCE_OVERWRITE = True

PROJECT_ROOT = Path("..").resolve()
DATA_DIR = Path("../../../../TOSICA-main/data/liver_cancer").resolve()
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "liver_cancer_data_preparation"
SOURCE_H5AD = DATA_DIR / "all_hvg_3000.h5ad"
PATH_FILE = "../KSPADA/resources/output_path4.txt"


TRAIN_SAMPLES = ["4", "5", "6", "7", "8", "9", "10", "11",
                 "12", "13", "14", "15"]
VALIDATION_SAMPLES = ["16"]
QUERY_SAMPLES = ["17", "18", "19", "20", "21"]

## Step 1 — Validate the source data

The source AnnData object must contain spatial coordinates, sample identifiers, and training labels.

In [2]:
source_adata = ad.read_h5ad(SOURCE_H5AD)
required_columns = {"x", "y", "batch", "Celltype"}
missing_columns = required_columns.difference(source_adata.obs.columns)
if missing_columns:
    raise KeyError(f"Missing required obs columns: {sorted(missing_columns)}")

source_adata.obs["batch"] = source_adata.obs["batch"].astype(str)
requested_samples = set(TRAIN_SAMPLES + VALIDATION_SAMPLES + QUERY_SAMPLES)
missing_samples = requested_samples.difference(source_adata.obs["batch"].unique())
if missing_samples:
    raise ValueError(f"Missing requested samples: {sorted(missing_samples)}")

/home/lixiangyu/anaconda3/envs/KSPADA/lib/python3.8/site-packages/anndata/_core/anndata.py:1838: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


## Step 2 — Split reference and query data

In [3]:
train_valid_adata = source_adata[
    source_adata.obs["batch"].isin(TRAIN_SAMPLES + VALIDATION_SAMPLES)
].copy()
query_adata = source_adata[source_adata.obs["batch"].isin(QUERY_SAMPLES)].copy()

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
train_valid_source = OUTPUT_DIR / "reference_source.h5ad"
query_source = OUTPUT_DIR / "query_source.h5ad"
KSPADA.write_adata(train_valid_adata, train_valid_source, force=FORCE_OVERWRITE)
KSPADA.write_adata(query_adata, query_source, force=FORCE_OVERWRITE)

/home/lixiangyu/anaconda3/envs/KSPADA/lib/python3.8/site-packages/anndata/_core/anndata.py:1838: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


PosixPath('/home/lixiangyu/sh/program/K-SPADA/outputs/liver_cancer_data_preparation/query_source.h5ad')

## Step 3 — Build path features and ligand-receptor-informed similarity graphs

In [ ]:
# Standardize the observation fields expected by K-SPADA, then align both datasets to the same genes.
reference_standardized = KSPADA.standardize_obs_columns(train_valid_source, x_key="x", y_key="y", batch_key="batch", label_key="Celltype")
query_standardized = KSPADA.standardize_obs_columns(query_source, x_key="x", y_key="y", batch_key="batch", label_key="Celltype", allow_missing_label=True)

reference_aligned, query_aligned = KSPADA.align_adatas_to_union_genes(
    [reference_standardized, query_standardized]
)

# Normalize each cell to the workflow's default total of 1e4, then apply log1p.
reference_normalized = KSPADA.normalize_total_log1p(reference_aligned, copy=False)
query_normalized = KSPADA.normalize_total_log1p(query_aligned, copy=False)

reference_norm_path = OUTPUT_DIR / "reference_data.h5ad"
query_norm_path = OUTPUT_DIR / "query_data.h5ad"
KSPADA.write_adata(reference_normalized, reference_norm_path, force=FORCE_OVERWRITE)
KSPADA.write_adata(query_normalized, query_norm_path, force=FORCE_OVERWRITE)

# Write one normalized file per biological sample for graph construction.
train_batches = [str(sample) for sample in TRAIN_SAMPLES]
valid_batches = [str(sample) for sample in VALIDATION_SAMPLES]
query_batches = [str(sample) for sample in QUERY_SAMPLES]
reference_split_prefix = OUTPUT_DIR / "reference_data_batch_"
query_split_prefix = OUTPUT_DIR / "query_data_batch_"
reference_split_paths = KSPADA.split_by_batch(
    reference_normalized,
    batch_ids=sorted(set(train_batches + valid_batches)),
    output_prefix=reference_split_prefix,
    force=FORCE_OVERWRITE,
)
query_split_paths = KSPADA.split_by_batch(
    query_normalized,
    batch_ids=query_batches,
    output_prefix=query_split_prefix,
    force=FORCE_OVERWRITE,
)

# Add path scores and a ligand-receptor-informed similarity graph to each sample, using the workflow defaults.
reference_graph_paths = {}
query_graph_paths = {}
for sample, split_path in reference_split_paths.items():
    graph_path = split_path.with_name(f"{split_path.stem}_path_scores_graph_k_{COMMUNICATION_GRAPH_NEIGHBORS}.h5ad")
    reference_graph_paths[sample] = KSPADA.prepare_query_cgraph(
        split_path, graph_path, k=COMMUNICATION_GRAPH_NEIGHBORS, force=FORCE_OVERWRITE
    )
for sample, split_path in query_split_paths.items():
    graph_path = split_path.with_name(f"{split_path.stem}_path_scores_graph_k_{COMMUNICATION_GRAPH_NEIGHBORS}.h5ad")
    query_graph_paths[sample] = KSPADA.prepare_query_cgraph(
        split_path, graph_path, k=COMMUNICATION_GRAPH_NEIGHBORS, force=FORCE_OVERWRITE
    )

train_valid_prefix = OUTPUT_DIR / f"reference_data_train_{'_'.join(train_batches)}_valid_{'_'.join(valid_batches)}"
train_valid_h5ad = KSPADA.create_train_valid_cgraph(
    train_batches=train_batches,
    valid_batches=valid_batches,
    batch_file_prefix=reference_split_prefix,
    train_adata_path=reference_norm_path,
    output_prefix=train_valid_prefix,
    k=COMMUNICATION_GRAPH_NEIGHBORS,
    force=FORCE_OVERWRITE,
)

## Notes

- Keep biological samples separate when assigning training, validation, and query data.
- Replace the sample names with values present in `adata.obs["batch"]`.
- This notebook documents generic data preparation. Tutorials 02–05 use their own existing prepared graphs; their execution is independent of this example.
